In [1]:
import math
import random
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


def set_plot_style():
    """A clean, paper-like matplotlib style (Nature-ish)."""

    palette = [
        "#1F77B4",
        "#FF7F0E",
        "#2CA02C",
        "#D62728",
        "#9467BD",
        "#8C564B",
        "#E377C2",
        "#7F7F7F",
        "#BCBD22",
        "#17BECF",
    ]

    plt.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.sans-serif": ["Arial", "DejaVu Sans"],
            "svg.fonttype": "none",
            "font.size": 13,
            "axes.spines.right": False,
            "axes.spines.top": False,
            "axes.linewidth": 1.5,
            "axes.grid": False,
            "axes.prop_cycle": plt.cycler(color=palette),
            "xtick.direction": "out",
            "ytick.direction": "out",
            "lines.linewidth": 2.0,
            "figure.dpi": 120,
            "savefig.dpi": 300,
            "savefig.bbox": "tight",
        }
    )


set_plot_style()

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
try:
    torch.set_num_threads(1)
except Exception:
    pass

print(f"PyTorch version: {torch.__version__}")
print(f"Default device: {'cuda' if torch.cuda.is_available() else 'cpu'}")


PyTorch version: 2.6.0+cu124
Default device: cuda


## 2 序列模型
## 2.1 理论计算题

字符序列 `"ababc"`，一阶马尔可夫模型 $p(x_t\mid x_{t-1})$，词汇表 $V=\{a,b,c\}$，拉普拉斯平滑（加 1 平滑）：

$$
p(x'\mid x)=\frac{\mathrm{count}(x\to x')+1}{\mathrm{count}(x)+|V|}.
$$

统计转移（由 `"ababc"` 得到）：

| 转移 | 次数 |
|------|------|
| $a\to b$ | 2 |
| $b\to a$ | 1 |
| $b\to c$ | 1 |

因此 $\mathrm{count}(b)=2$。

### 1. $p(a\mid b)$

$$
p(a\mid b)=\frac{1+1}{2+3}=\boxed{\frac{2}{5}=0.4}.
$$

### 2. $p(c\mid b)$

$$
p(c\mid b)=\frac{1+1}{2+3}=\boxed{\frac{2}{5}=0.4}.
$$


## 2.2 编程题

实现 `preprocess_text(text, n)`：小写化、去标点、分词、按词频构建词表（ID 从 0 开始），并用长度为 $n$ 的滑动窗口生成自回归语言模型的特征与标签。


In [2]:
def preprocess_text(text, n):
    """文本预处理：词表构建 + n-gram 特征/标签生成。"""
    if n < 1:
        raise ValueError("n must be >= 1")

    cleaned = []
    for ch in text.lower():
        if ch.isalpha() or ch.isspace():
            cleaned.append(ch)
    tokens = " ".join("".join(cleaned).split()).split()

    freq = Counter(tokens)
    vocab = {word: idx for idx, (word, _) in enumerate(freq.most_common())}

    features, labels = [], []
    for i in range(len(tokens) - n + 1):
        context = tokens[i : i + n]
        next_word = tokens[i + n] if i + n < len(tokens) else None
        features.append(context)
        labels.append(next_word)

    return vocab, (features, labels)


text = "The time machine"
n = 2
vocab, (features, labels) = preprocess_text(text, n)

print("Vocabulary (word -> id):")
for word, idx in vocab.items():
    print(f"  {word!r}: {idx}")
print(f"\nFeatures: {features}")
print(f"Labels:   {labels}")

train_pairs = [(f, y) for f, y in zip(features, labels) if y is not None]
print(f"\nUsable training pairs ({len(train_pairs)}):")
for f, y in train_pairs:
    print(f"  {f} -> {y!r}")


Vocabulary (word -> id):
  'the': 0
  'time': 1
  'machine': 2

Features: [['the', 'time'], ['time', 'machine']]
Labels:   ['machine', None]

Usable training pairs (1):
  ['the', 'time'] -> 'machine'


## 3 循环神经网络
## 3.1 理论计算题

线性 RNN（无偏置）：

$$
h_t = W_{hh} h_{t-1} + W_{hx} x_t, \qquad o_t = W_{oh} h_t,
$$

平方损失

$$
L = \frac{1}{2}\sum_{t=1}^{T}(o_t-y_t)^2.
$$

### 对 $W_{hh}$ 的梯度（BPTT）

先对输出层：

$$
\frac{\partial L}{\partial o_t} = o_t - y_t, \qquad
\frac{\partial L}{\partial h_t} = W_{oh}^\top (o_t-y_t) + \frac{\partial L}{\partial h_{t+1}}\, W_{hh}^\top.
$$

由于 $h_t = W_{hh} h_{t-1} + W_{hx} x_t$，有 $\partial h_t / \partial W_{hh} = h_{t-1}^\top$（按外积形式），因此

$$
\boxed{
\frac{\partial L}{\partial W_{hh}}
= \sum_{t=1}^{T} \frac{\partial L}{\partial h_t}\, h_{t-1}^\top
}.
$$

展开到所有时间步，$\partial L/\partial h_t$ 含从 $T$ 到 $t$ 的连乘项：

$$
\frac{\partial L}{\partial h_t}
= \sum_{k=t}^{T} \left(\prod_{j=t+1}^{k} W_{hh}^\top\right) W_{oh}^\top (o_k-y_k).
$$

### 梯度消失 / 爆炸条件

- 若 $W_{hh}$ 的最大奇异值（或谱半径）$\rho(W_{hh}) < 1$，则连乘项随时间步指数衰减，出现**梯度消失**。
- 若 $\rho(W_{hh}) > 1$，连乘项指数放大，出现**梯度爆炸**。
- 当 $T$ 很大且 $\rho(W_{hh})\approx 1$ 时，梯度仍可能极不稳定。


## 3.2 编程题

实现带 `tanh` 激活的 RNN 单元前向传播与单步反向传播（只计算梯度，不更新参数）。


In [3]:
def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """RNN 单步前向：h_t = tanh(x_t W_hx^T + h_prev W_hh^T + b_h)."""
    z_t = x_t @ W_hx.T + h_prev @ W_hh.T + b_h
    h_t = torch.tanh(z_t)
    cache = (x_t, h_prev, z_t, h_t, W_hx, W_hh)
    return h_t, cache


def rnn_cell_backward(dh_next, cache):
    """RNN 单步反向：已知 dL/dh_t，返回其余梯度。"""
    x_t, h_prev, z_t, h_t, W_hx, W_hh = cache

    dz_t = dh_next * (1.0 - h_t ** 2)
    dW_hx = dz_t.T @ x_t
    dW_hh = dz_t.T @ h_prev
    db_h = dz_t.sum(dim=0)
    dx_t = dz_t @ W_hx
    dh_prev = dz_t @ W_hh
    return dx_t, dh_prev, dW_hx, dW_hh, db_h


torch.manual_seed(SEED)
batch_size, input_size, hidden_size = 2, 3, 4

x_t = torch.randn(batch_size, input_size, requires_grad=True)
h_prev = torch.randn(batch_size, hidden_size, requires_grad=True)
W_hx = torch.randn(hidden_size, input_size, requires_grad=True)
W_hh = torch.randn(hidden_size, hidden_size, requires_grad=True)
b_h = torch.zeros(hidden_size, requires_grad=True)

h_t, cache = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
h_t.retain_grad()
loss = h_t.pow(2).sum()
loss.backward()

manual_dx, manual_dh_prev, manual_dW_hx, manual_dW_hh, manual_db_h = rnn_cell_backward(
    h_t.grad, cache
)

checks = {
    "dx_t": torch.allclose(manual_dx, x_t.grad, atol=1e-5),
    "dh_prev": torch.allclose(manual_dh_prev, h_prev.grad, atol=1e-5),
    "dW_hx": torch.allclose(manual_dW_hx, W_hx.grad, atol=1e-5),
    "dW_hh": torch.allclose(manual_dW_hh, W_hh.grad, atol=1e-5),
    "db_h": torch.allclose(manual_db_h, b_h.grad, atol=1e-5),
}

print(f"h_t shape: {tuple(h_t.shape)}")
for name, ok in checks.items():
    print(f"manual backward matches autograd for {name}: {ok}")


h_t shape: (2, 4)
manual backward matches autograd for dx_t: True
manual backward matches autograd for dh_prev: True
manual backward matches autograd for dW_hx: True
manual backward matches autograd for dW_hh: True
manual backward matches autograd for db_h: True


## 4 高级循环神经网络
## 4.1 理论计算题

深度**双向** RNN：$L$ 层，每层隐藏单元数 $H$，输入维度 $D$，最终输出层维度 $O$（仅计最后输出层，忽略嵌入层与额外投影）。

PyTorch 风格下，每一层每个方向含 $W_{ih}, W_{hh}$ 及两组偏置，参数为

$$
\underbrace{H\cdot d_{\text{in}} + H^2 + 2H}_{\text{单方向单层}}.
$$

- 第 1 层（输入 $D$）：每层双向共 $2(HD + H^2 + 2H)$。
- 第 $l$ 层（$l\ge 2$，输入为上一层拼接后的 $2H$）：每层双向共 $2(2H^2 + H^2 + 2H)=6H^2+4H$。
- 输出层：$W_o\in\mathbb{R}^{O\times 2H},\, b_o\in\mathbb{R}^{O}$，共 $2HO+O$。

总参数量：

$$
\boxed{
N_{\text{params}}
= 2HO + O + 2(HD + H^2 + 2H) + (L-1)(6H^2 + 4H)
= 2HO + O + 2HD + (6L-4)H^2 + 4LH.
}$$


## 4.2 编程题

实现双向 RNN 编码器，返回每个时间步拼接的前向/后向隐藏状态，以及最终时间步的序列表示。


In [4]:
class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=False,
            bidirectional=True,
        )

    def forward(self, X):
        """X: (seq_len, batch, input_dim)."""
        outputs, h_n = self.rnn(X)
        # outputs: (seq_len, batch, 2*hidden_dim)
        final_repr = outputs[-1]
        return outputs, final_repr


torch.manual_seed(SEED)
seq_len, batch_size, input_dim, hidden_dim = 5, 3, 4, 6
X = torch.randn(seq_len, batch_size, input_dim)

encoder = BiRNNEncoder(input_dim=input_dim, hidden_dim=hidden_dim, num_layers=2)
outputs, final_repr = encoder(X)

print(f"Input shape:           {tuple(X.shape)}")
print(f"All-step hidden shape: {tuple(outputs.shape)}")
print(f"Final-step repr shape: {tuple(final_repr.shape)}")

# 与 torch.nn.RNN 直接输出对照
rnn_ref = nn.RNN(input_dim, hidden_dim, num_layers=2, bidirectional=True)
rnn_ref.load_state_dict(encoder.rnn.state_dict())
out_ref, _ = rnn_ref(X)
print(f"max |diff| vs nn.RNN: {(outputs - out_ref).abs().max().item():.3e}")


Input shape:           (5, 3, 4)
All-step hidden shape: (5, 3, 12)
Final-step repr shape: (3, 12)
max |diff| vs nn.RNN: 0.000e+00


## 5 嵌入向量
## 5.1 理论计算题

Skip-gram + 负采样（$K$ 个负样本）。中心词 $w_c$ 向量 $v_c$，上下文词 $w_o$ 向量 $u_o$，负样本 $w_k$ 向量 $u_{w_k}$。

目标函数（最大化对数似然，等价于最小化下式损失）：

$$
\boxed{
\mathcal{L}
= -\log \sigma(u_o^\top v_c)
  - \sum_{k=1}^{K} \log \sigma\left(-u_{w_k}^\top v_c\right)
},
$$

其中 $\sigma(\cdot)$ 为 sigmoid。

**负样本采样**：从噪声分布 $P_n(w)$ 中独立采样 $K$ 个词作为负样本，常取

$$
P_n(w) \propto U(w)^{3/4},
$$

$U(w)$ 为语料中词 $w$ 的 unigram 频率。按 $P_n(w)$ 做多项式采样即可得到 $\{w_k\}_{k=1}^K$。


## 5.2 编程题

实现 CBOW 前向传播与完整 softmax 交叉熵损失（不使用负采样）。


In [5]:
def cbow_loss(context_indices, center_indices, W, W_out):
    """CBOW 损失：平均上下文嵌入 -> 线性输出 -> softmax CE。

    参数
    ----
    context_indices : LongTensor, shape (batch, context_size)
    center_indices  : LongTensor, shape (batch,)
    W               : Tensor, shape (V, d)
    W_out           : Tensor, shape (d, V)
    """
    context_emb = W[context_indices].mean(dim=1)
    logits = context_emb @ W_out
    return F.cross_entropy(logits, center_indices)


torch.manual_seed(SEED)
V, d, context_size, batch_size = 20, 8, 3, 4
W = torch.randn(V, d)
W_out = torch.randn(d, V)

context_indices = torch.randint(0, V, (batch_size, context_size))
center_indices = torch.randint(0, V, (batch_size,))

loss = cbow_loss(context_indices, center_indices, W, W_out)
print(f"V={V}, d={d}, context_size={context_size}, batch_size={batch_size}")
print(f"CBOW cross-entropy loss: {loss.item():.6f}")

# 与逐样本手算平均对照
manual_losses = []
for i in range(batch_size):
    hidden = W[context_indices[i]].mean(dim=0)
    logits = hidden @ W_out
    manual_losses.append(F.cross_entropy(logits.unsqueeze(0), center_indices[i : i + 1]))
manual_mean = torch.stack(manual_losses).mean()
print(f"manual mean loss check: {torch.allclose(loss, manual_mean, atol=1e-5)}")


V=20, d=8, context_size=3, batch_size=4
CBOW cross-entropy loss: 2.754257
manual mean loss check: True


## 6 注意力机制
## 6.1 理论计算题

给定 $Q\in\mathbb{R}^{2\times 4}$，$K\in\mathbb{R}^{3\times 4}$，$V\in\mathbb{R}^{3\times 5}$，$d_k=4$。

**Step 1：得分矩阵**

$$
S = \frac{QK^\top}{\sqrt{d_k}} = \frac{QK^\top}{2} \in \mathbb{R}^{2\times 3}.
$$

**Step 2：注意力权重**

对每一行做 softmax：

$$
A = \mathrm{softmax}(S,\ \text{dim}=-1) \in \mathbb{R}^{2\times 3}.
$$

**Step 3：加权求和**

$$
\mathrm{Output} = AV \in \mathbb{R}^{2\times 5}.
$$

下面用一组具体数值演示（与符号推导一致）。


In [6]:
torch.manual_seed(SEED)
Q = torch.tensor(
    [
        [1.0, 0.0, 1.0, 0.0],
        [0.0, 1.0, 1.0, 1.0],
    ]
)
K = torch.tensor(
    [
        [1.0, 0.0, 0.0, 1.0],
        [0.0, 1.0, 0.0, 1.0],
        [1.0, 1.0, 0.0, 0.0],
    ]
)
V = torch.tensor(
    [
        [1.0, 2.0, 0.0, 1.0, 0.0],
        [0.0, 1.0, 1.0, 0.0, 2.0],
        [2.0, 0.0, 1.0, 1.0, 1.0],
    ]
)

dk = Q.shape[-1]
scores = Q @ K.T / math.sqrt(dk)
attn = torch.softmax(scores, dim=-1)
output = attn @ V

print("Q shape:", tuple(Q.shape))
print("K shape:", tuple(K.shape))
print("V shape:", tuple(V.shape))
print("\nScores S = QK^T / sqrt(dk):")
print(scores)
print("\nAttention weights A = softmax(S):")
print(attn)
print("\nOutput = A V:")
print(output)


Q shape: (2, 4)
K shape: (3, 4)
V shape: (3, 5)

Scores S = QK^T / sqrt(dk):
tensor([[0.5000, 0.0000, 0.5000],
        [0.5000, 1.0000, 0.5000]])

Attention weights A = softmax(S):
tensor([[0.3837, 0.2327, 0.3837],
        [0.2741, 0.4519, 0.2741]])

Output = A V:
tensor([[1.1510, 1.0000, 0.6163, 0.7673, 0.8490],
        [0.8222, 1.0000, 0.7259, 0.5481, 1.1778]])


## 6.2 编程题

实现 Multi-Head Attention 前向传播（`num_heads=2`, `d_model=4`）。


In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, X):
        """X: (seq_len, batch, d_model)."""
        seq_len, batch_size, _ = X.shape

        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)

        def split_heads(tensor):
            return tensor.view(seq_len, batch_size, self.num_heads, self.d_k).permute(1, 2, 0, 3)

        Q = split_heads(Q)
        K = split_heads(K)
        V = split_heads(V)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        attn = torch.softmax(scores, dim=-1)
        context = attn @ V

        context = context.permute(2, 0, 1, 3).contiguous().view(seq_len, batch_size, self.d_model)
        return self.W_o(context)


torch.manual_seed(SEED)
seq_len, batch_size, d_model, num_heads = 4, 2, 4, 2
X = torch.randn(seq_len, batch_size, d_model)

mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
Y = mha(X)

print(f"Input shape:  {tuple(X.shape)}")
print(f"Output shape: {tuple(Y.shape)}")
assert Y.shape == X.shape
print("Shape check passed.")


Input shape:  (4, 2, 4)
Output shape: (4, 2, 4)
Shape check passed.
